# 1. Librairies et paramètres

In [7]:
# année à traiter
year = "2026"

# galerie à enlever si déjà traitée
# exemple : galerie_to_remove =  ["10 km Moulin bleu", "10 km Rue des Sports", "Autour de la course", "L'arrivée", "Podiums"]
galerie_to_remove = ["Galerie_test_2"]

In [13]:
import os
from pathlib import Path
import time
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np
import cv2
import gc
from PIL import Image
from paddleocr import PaddleOCR

In [2]:
# Fonction pour détection de dossart 
ocr = PaddleOCR(use_textline_orientation=True, lang='en')

c:\Users\lemer\anaconda3\envs\paddleocr\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:717: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lemer\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lemer\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lemer\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', No

# 2. Fonction pour extraire les dossards

In [5]:
def detection_dossart(files, galerie_path): 

    start = time.time() 
    detection_results = []

    # parcours des images 
    for i, file in enumerate(files):
        print(f"Image : {file}")

        # redimensionnement de l'image 
        image_path = os.path.join(galerie_path, file)
        img = cv2.imread(image_path)

        # lecture de l'image si cv2 ne fonctionne pas 
        if img is None:
            try:
                pil_img = Image.open(image_path).convert("RGB")
                img = np.array(pil_img)
                img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            except Exception as e:
                print(f"Impossile de lire {image_path} : {e}")
                continue

        max_side = 2000
        height, width = img.shape[:2]
        scale = min(max_side/height, max_side/width, 1.0)
        if scale < 1.0:
            img = cv2.resize(img, (int(width*scale), int(height*scale)))
        
        # execution paddleOCR
        results = ocr.predict(img)

        # Si aucun texte n'est détecté 
        if not results or all(line is None for line in results):

            # créer une liste detection et ajout a detection_results
            detection = []
            detection.append(file)
            for _ in range(10):
                detection.append(None)
            detection_results.append(detection)
            continue

        # Si du texte est détecté 
        if results:
            detect_digit = False
            for page in results:
                texts = page['rec_texts']
                scores = page['rec_scores']
                polys = page['rec_polys']

                # Parcours de chaque couple text-score-coordonnées
                for text, score, poly in zip(texts, scores, polys):
                    # vérification qu'on détecte des nombres
                    if text.isdigit(): 
                        detect_digit = True
                        detection = []
                        detection.append(file)
                        detection.append(text)
                        detection.append(score)
                        detection.extend(poly.flatten().tolist())
                        detection_results.append(detection)

            if not detect_digit:
                # créer une liste detection et ajout a detection_results
                detection = []
                detection.append(file)
                for _ in range(10):
                    detection.append(None)
                detection_results.append(detection)

        # libération mémoire
        del results
        gc.collect()
        

    print(len(detection_results))

    end = time.time() 
    print(f"Time : {end - start}") 

    return detection_results

In [14]:
# récupération de toutes les galeries de photos
images_repertory = Path('Images') / year
galeries = [d for d in os.listdir(images_repertory) if os.path.isdir(os.path.join(images_repertory, d))]

# enlever les galeries déjà traitées
for tr in galerie_to_remove:
    if tr in galeries:
        galeries.remove(tr)
print(galeries)

['Galerie_test']


**Application de la fonction**

In [15]:
for galerie in galeries:


    # détection des dossarts

    # Lister les fichiers image dans le dossier
    galerie_path = Path(images_repertory) / galerie
    valids_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')
    files = sorted([f for f in os.listdir(galerie_path) if f.lower().endswith(valids_extensions)])
    
    # détection des dossarts
    detection_results = detection_dossart(files, galerie_path)


    # Sauvegarde des résultats

    # répertoire où sont sauvegardés les résultats
    folder_result = Path("Results") / year / galerie
    os.makedirs(folder_result, exist_ok=True)
    result_file_name = "detection.csv"
    result_file_path = Path(folder_result) / result_file_name

    # Définition des en-têtes
    headers = ["file", "text", "score", "x1", "y1", "x2", "y2", "x3", "y3", "x4", "y4"]

    df = pd.DataFrame(detection_results, columns=headers)
    df.to_csv(result_file_path, index=False, encoding='utf-8')

    print(f'Fichier enregistré : {result_file_path}')
    print(os.listdir(folder_result))

print(f"\nNombre de dossards détectés : {len(detection_results)}")

Image : 1.jpg
Image : IMG_4848.JPG
4
Time : 22.174166202545166
Fichier enregistré : Results\2026\Galerie_test\detection.csv
['detection.csv', 'detection_post_process.csv', 'detection_v1_post_process.csv']

Nombre de dossards détectés : 4
